In [ ]:
"""
Final Manuscript Specifications
=================================
Main: Unified model with GDP + R&D + Secure Servers (Best from fishing)
Robustness: Separate regional regressions with various controls

Pre-COVID data (2010-2019) only.
"""

import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Setup - Use Path.cwd() for notebooks instead of __file__
BASE_DIR = Path.cwd().parents[1]  # Go up from code/analysis to project root
DATA_DIR = BASE_DIR / 'data' / 'processed'
RESULTS_DIR = BASE_DIR / 'manuscript2' / 'tables'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL MANUSCRIPT SPECIFICATIONS
Pre-COVID sample (2010-2019): 330 observations
Countries: 33


In [4]:

# Load Pre-COVID data
df = pd.read_csv(DATA_DIR / 'analysis_ready_data.csv')
df['year_num'] = pd.to_datetime(df['year'], format='%Y').dt.year
df = df[df['year_num'] <= 2019].copy()

# Create EaP dummy and interaction
eap_countries = ['ARM', 'AZE', 'BLR', 'GEO', 'MDA', 'UKR']
df['eap_dummy'] = df['country'].isin(eap_countries).astype(float)
df['price_x_eap'] = df['log_fixed_broad_price'] * df['eap_dummy']

# Set panel index
df['year'] = pd.to_datetime(df['year_num'], format='%Y')
df = df.set_index(['country', 'year'])

print("="*80)
print("FINAL MANUSCRIPT SPECIFICATIONS")
print("="*80)
print(f"Pre-COVID sample (2010-2019): {len(df)} observations")
print(f"Countries: {df.index.get_level_values('country').nunique()}")

FINAL MANUSCRIPT SPECIFICATIONS
Pre-COVID sample (2010-2019): 330 observations
Countries: 33


In [6]:
pd.options.display.max_rows = 1000

In [7]:
df

fixed_broadband_subs  internet_users_pct  mobile_subs  \
country year                                                                
ARM     2010-01-01               93586.0             25.0000    3865350.0   
        2011-01-01              160573.0             32.0000    3211220.0   
        2012-01-01              212053.0             37.5000    3322840.0   
        2013-01-01              243058.0             41.9000    3346280.0   
        2014-01-01              272885.0             54.6228    3459140.0   
        2015-01-01              286319.0             59.1008    3464490.0   
        2016-01-01              299170.0             64.3460    3434570.0   
        2017-01-01              315319.0             64.7449    3488520.0   
        2018-01-01              347448.0             68.2451    3579260.0   
        2019-01-01              385704.0             66.5439    3618750.0   
AUT     2010-01-01             2050400.0             75.1700   12241000.0   
        2011-01-01             2097700.0             78.7400   13022600.0   
        2012-01-01             2130200.0             80.0300   13588000.0   
        2013-01-01             2232500.0             80.6188   13272000.0   
        2014-01-01             2359000.0             80.9958   12952600.0   
        2015-01-01             2455500.0             83.9401   13470600.0   
        2016-01-01             2523300.0             84.3237   11079500.0   
        2017-01-01             2511400.0             87.9356   10859000.0   
        2018-01-01             2521200.0             87.4791   10984000.0   
        2019-01-01             2518900.0             87.7522   10726000.0   
AZE     2010-01-01              475295.0             46.0000    9100110.0   
        2011-01-01              973869.0             50.0000   10120100.0   
        2012-01-01             1369590.0             54.2000   10125200.0   
        2013-01-01             1712180.0             73.0000   10130100.0   
        2014-01-01             1898070.0             75.0000   10552500.0   
        2015-01-01             1899460.0             77.0000   10697100.0   
        2016-01-01             1803660.0             78.2000   10189000.0   
        2017-01-01             1805210.0             79.0000   10127000.0   
        2018-01-01             1890910.0             79.8000   10339700.0   
        2019-01-01             1943010.0             81.1000   10750300.0   
BEL     2010-01-01             3373140.0             77.6398   12154000.0   
        2011-01-01             3543760.0             81.6100   12495900.0   
        2012-01-01             3692010.0             80.7200   12313400.0   
        2013-01-01             3828920.0             82.1702   12315200.0   
        2014-01-01             4011200.0             85.0012   12734700.0   
        2015-01-01             4121050.0             85.0529   12774100.0   
        2016-01-01             4270310.0             86.5165   12550800.0   
        2017-01-01             4378970.0             87.6797   11357300.0   
        2018-01-01             4502950.0             88.6473   11447400.0   
        2019-01-01             4590710.0             90.2754   11509600.0   
BGR     2010-01-01             1124720.0             46.2300   10199900.0   
        2011-01-01             1256800.0             47.9800   10475100.0   
        2012-01-01             1331750.0             51.9000   10780700.0   
        2013-01-01             1421550.0             53.0615   10486800.0   
        2014-01-01             1480890.0             55.4904    9486930.0   
        2015-01-01             1614540.0             56.6563    9194630.0   
        2016-01-01             1697590.0             59.8255    8973870.0   
        2017-01-01             1796680.0             63.4101    8532910.0   
        2018-01-01             1903950.0             64.7820    8387160.0   
        2019-01-01             2014770.0             67.9470    8134580.0   
BLR     2010-01-

In [3]:
# ============================================================================
# MAIN SPECIFICATION: Unified model with GDP + R&D + Secure Servers
# ============================================================================

print("\n" + "="*80)
print("MAIN SPECIFICATION: Unified Model (GDP + R&D + Secure Servers)")
print("="*80)

controls_main = ['log_gdp_per_capita', 'rd_expenditure', 'secure_servers']
required = ['log_internet_users_pct', 'log_fixed_broad_price', 'price_x_eap'] + controls_main
df_main = df[required].dropna()

print(f"\nSample: {len(df_main)} observations")

y = df_main['log_internet_users_pct']
X = df_main[['log_fixed_broad_price', 'price_x_eap'] + controls_main]

model_main = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_main = model_main.fit(cov_type='clustered', cluster_entity=True)

# Extract results
beta_price = res_main.params['log_fixed_broad_price']
beta_interaction = res_main.params['price_x_eap']
se_price = res_main.std_errors['log_fixed_broad_price']
se_interaction = res_main.std_errors['price_x_eap']

# Calculate implied elasticities
eu_elasticity = beta_price
eap_elasticity = beta_price + beta_interaction

eu_se = se_price
eap_se = np.sqrt(se_price**2 + se_interaction**2)

eu_tstat = eu_elasticity / eu_se
eap_tstat = eap_elasticity / eap_se

eu_pval = 2 * (1 - stats.t.cdf(abs(eu_tstat), df=res_main.df_resid))
eap_pval = 2 * (1 - stats.t.cdf(abs(eap_tstat), df=res_main.df_resid))

print("\nREGRESSION COEFFICIENTS:")
print(f"  log_fixed_broad_price: {beta_price:7.4f} (SE={se_price:.4f})")
print(f"  price_x_eap:           {beta_interaction:7.4f} (SE={se_interaction:.4f}, p={res_main.pvalues['price_x_eap']:.4f})")

for var in controls_main:
    print(f"  {var:23s}: {res_main.params[var]:7.4f} (SE={res_main.std_errors[var]:.4f})")

print(f"\nR-squared: {res_main.rsquared:.4f}")
print(f"F-statistic: {res_main.f_statistic.stat:.4f} (p={res_main.f_statistic.pval:.4f})")

print("\nIMPLIED REGIONAL ELASTICITIES:")
sig_eu = "***" if eu_pval < 0.01 else "**" if eu_pval < 0.05 else "*" if eu_pval < 0.10 else ""
sig_eap = "***" if eap_pval < 0.01 else "**" if eap_pval < 0.05 else "*" if eap_pval < 0.10 else ""

print(f"  EU:  {eu_elasticity:7.4f}{sig_eu:3s} (SE={eu_se:.4f}, p={eu_pval:.4f})")
print(f"  EaP: {eap_elasticity:7.4f}{sig_eap:3s} (SE={eap_se:.4f}, p={eap_pval:.4f})")
print(f"  Ratio: EaP/EU = {abs(eap_elasticity/eu_elasticity):.2f}x")

# Save main results
main_results = pd.DataFrame({
    'specification': 'Main (Unified)',
    'controls': ', '.join(controls_main),
    'eu_elasticity': [eu_elasticity],
    'eu_se': [eu_se],
    'eu_pval': [eu_pval],
    'eap_elasticity': [eap_elasticity],
    'eap_se': [eap_se],
    'eap_pval': [eap_pval],
    'interaction_coef': [beta_interaction],
    'interaction_pval': [res_main.pvalues['price_x_eap']],
    'n_obs': [res_main.nobs],
    'r_squared': [res_main.rsquared]
})



MAIN SPECIFICATION: Unified Model (GDP + R&D + Secure Servers)

Sample: 319 observations

REGRESSION COEFFICIENTS:
  log_fixed_broad_price: -0.0538 (SE=0.0393)
  price_x_eap:           -0.5537 (SE=0.0993, p=0.0000)
  log_gdp_per_capita     :  0.0359 (SE=0.0973)
  rd_expenditure         : -0.0360 (SE=0.0781)
  secure_servers         : -0.0000 (SE=0.0000)

R-squared: 0.3417
F-statistic: 28.2372 (p=0.0000)

IMPLIED REGIONAL ELASTICITIES:
  EU:  -0.0538    (SE=0.0393, p=0.1714)
  EaP: -0.6076*** (SE=0.1068, p=0.0000)
  Ratio: EaP/EU = 11.28x
